# TED 01: seleção e inspeção da base

Base: [Brazilian Legal Proceedings](https://www.kaggle.com/datasets/felipepolo/brazilian-legal-proceedings), de Felipe Maia Polo, versão 2 (14/05/2021). A etapa está descrita nas seções 1 a 5 de `docs/JuriPredict_Documento_Tecnico_BigData.pdf`.

Este notebook executa `src/ted01_baixar_e_inspecionar.py`, que baixa `labeled.json` e `unlabeled.json` para `data/raw/`, confere o SHA-256 da versão usada no documento e grava a inspeção em `resultados/ted01_inspecao.json`. Os dados não são versionados; os motivos estão em `data/README.md`.

In [1]:
import json, os, subprocess, sys, time
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

RAIZ = Path.cwd().parents[1]
RESULTADOS = RAIZ / "resultados"
pd.set_option("display.float_format", lambda x: f"{x:.0f}" if float(x).is_integer() else f"{x:.4f}")


def rodar(*args):
    """Executa um comando Python a partir da raiz do repositório e informa quanto tempo levou."""
    inicio = time.time()
    r = subprocess.run([sys.executable, *args], cwd=RAIZ, capture_output=True, text=True, encoding="utf-8",
                       env={**os.environ, "PYTHONIOENCODING": "utf-8", "PYTHONDONTWRITEBYTECODE": "1"})
    if r.returncode:
        raise RuntimeError(r.stderr[-3000:])
    print(f"{' '.join(args)}: {(time.time() - inicio) / 60:.1f} min")
    return r.stdout


def figuras(*nomes):
    for nome in nomes:
        display(Image(filename=RESULTADOS / "figuras" / f"{nome}.png"))

In [2]:
rodar("src/ted01_baixar_e_inspecionar.py");

src/ted01_baixar_e_inspecionar.py: 0.3 min


## Arquivos obtidos e integridade

In [3]:
inspecao = json.loads((RESULTADOS / "ted01_inspecao.json").read_text(encoding="utf-8"))
pd.DataFrame({"bytes": inspecao["bytes"], "sha256": inspecao["sha256"]})

,bytes,sha256
labeled.json,127627741,653013a741af31ad389f44458bc0bf5fd4c78a38064655...
unlabeled.json,530110583,0451bc19da272cd849a25e510d44ad1735f42a51436514...


## Volume e estrutura

In [4]:
r = inspecao["rotulado"]
pd.Series({"processos rotulados": r["processos"], "movimentações dos rotulados": r["movimentacoes"],
           "movimentações sem rótulo": inspecao["sem_rotulo"]["movimentacoes"],
           "textos distintos sem rótulo": inspecao["sem_rotulo"]["textos_distintos"],
           "textos vazios sem rótulo": inspecao["sem_rotulo"]["vazios"]}, name="quantidade")

processos rotulados               6449
movimentações dos rotulados     545843
movimentações sem rótulo       2904255
textos distintos sem rótulo    1305090
textos vazios sem rótulo            22
Name: quantidade, dtype: int64

In [5]:
print("atributos de cada processo rotulado:", r["atributos_por_processo"])
pd.Series(r["rotulos"], name="processos")

atributos de cada processo rotulado: ['label', 'texts']


H:Arquivado    3040
H:Ativo        2917
H:Suspenso      492
Name: processos, dtype: int64

In [6]:
pd.DataFrame({"movimentações por processo": r["movimentacoes_por_processo"],
              "caracteres por movimentação": r["caracteres_por_movimentacao"]})

,movimentações por processo,caracteres por movimentação
min,1,5
mediana,60,81
media,84.6400,188.8900
max,2659,160093
